<p class="lead">This notebook is the R edition of the first unit of the course. The lecture slides are a map; this text is the territory. It is written to be read linearly once, with every code chunk executed as you reach it, and then consulted by section. Unlike a textbook chapter, every figure and every number in it is produced by R code you can see and change: when a sentence says "the table shows 444 benign and 239 malignant cases", the table above it was computed in front of you. The unit has two halves. The first (Sections 1&ndash;3) answers <em>what artificial intelligence is</em>, where it came from, and which families of methods it comprises. The second (Sections 4&ndash;7) introduces <em>pattern recognition</em> &mdash; the discipline that most of this course belongs to &mdash; and the question of how data about people should be handled by the engineers who build systems that learn from them.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>State at least two operational definitions of artificial intelligence and explain why the field has no single agreed definition.</li>
<li>Place the main events of the history of AI on a timeline, name the people associated with them, and explain what caused the two "AI winters".</li>
<li>Distinguish the symbolic, connectionist, statistical and evolutionary approaches to AI and give one medical example of each; distinguish system, algorithm, model and equation; classify a system as narrow/general and as reactive/limited-memory.</li>
<li>Describe what generative AI is, which modalities exist, and which risks it adds to those of classification systems.</li>
<li>Define pattern recognition and describe its pipeline: sensing, representation, feature extraction, classification, decision.</li>
<li>Represent a patient, an image or a signal as a feature vector; classify data types (continuous, discrete, nominal, ordinal, binary) and explain why the type dictates the admissible operations.</li>
<li>Explain what a training sample is, why it is a sample and not the population, and why that distinction is the root of every failure mode studied later in the course.</li>
<li>Explain re-identification, k-anonymity, dataset shift and label bias, and name the Chilean and international regulations that govern health data.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets the output. Exercises (green boxes) ask you to modify the chunk above them and re-run; they are the fastest way to learn. Blue boxes carry key ideas, orange boxes carry common misconceptions, and purple boxes connect this unit to other units or to the bioinformatics course (BGM324). The notebook uses only base R, the <code>stats</code> package and the modelling packages <code>mlbench</code>, <code>rpart</code>, <code>rpart.plot</code> and <code>class</code>, all pre-installed in this environment and installable in one line in a fresh Google Colab R runtime (see below); no data is downloaded, because the datasets ship with the <code>mlbench</code> package<sup class="c"><a href="#r38">[38]</a></sup> and with R itself.<sup class="c"><a href="#r39">[39]</a></sup> Supplement S0 covers the R toolkit if you have not used these libraries before.</p></div>



In [ ]:
# Setup: libraries used throughout the notebook. The random seed is fixed so that every
# number quoted in the text is reproducible. In a fresh Colab R runtime, install once:
#   install.packages(c("mlbench","rpart","rpart.plot","class"))
suppressPackageStartupMessages({
  library(mlbench); library(rpart); library(rpart.plot); library(class)
})
set.seed(0)
cat(R.version.string, "| mlbench", as.character(packageVersion("mlbench")),
    "| rpart", as.character(packageVersion("rpart")), "\n")



The printout records the R version and package versions these notes were built with. `set.seed(0)` makes every later random split reproducible; where a chunk draws random numbers it re-seeds explicitly so that the number quoted in the surrounding text is the number the chunk produces.

## 1. What is artificial intelligence?

The term was coined in a funding proposal. In the summer of 1955, John McCarthy, Marvin Minsky, Nathaniel Rochester and Claude Shannon asked the Rockefeller Foundation to support a two-month workshop at Dartmouth College to study "the conjecture that every aspect of learning or any other feature of intelligence can in principle be so precisely described that a machine can be made to simulate it".<sup class="c"><a href="#r1">[1]</a></sup> That sentence contains the whole programme of the field and also its central ambiguity: it defines artificial intelligence by reference to <em>intelligence</em>, which itself has no agreed definition.

Five years earlier, Alan Turing had sidestepped the problem. Instead of asking "can machines think?", a question he considered "too meaningless to deserve discussion", he proposed an operational test: a machine is intelligent if an interrogator exchanging written messages cannot reliably tell it from a human.<sup class="c"><a href="#r2">[2]</a></sup> The imitation game &mdash; the Turing test &mdash; is still the best-known definition of AI outside the field, and still criticised inside it, because it measures the ability to <em>appear</em> human rather than the ability to solve problems.

### 1.1 Four ways to define the field

Russell and Norvig, whose textbook has been the standard reference for three decades, organise the historical definitions along two axes: whether the goal is to reproduce <em>human</em> behaviour or to achieve <em>rational</em> (ideal) behaviour, and whether the object of study is <em>thought</em> or <em>action</em>.<sup class="c"><a href="#r3">[3]</a></sup>

| | Thinking | Acting |
|---|---|---|
| **Humanly** | Cognitive modelling: build systems that think the way people do (cognitive science, neuroscience) | The Turing test: build systems whose behaviour is indistinguishable from a person's |
| **Rationally** | Laws of thought: formal logic, correct inference from premises | **Rational agents**: build systems that perceive their environment and act so as to maximise the expected achievement of their goals |

Modern AI sits almost entirely in the bottom-right cell. A <strong>rational agent</strong> is anything that perceives its environment through sensors and acts on it through actuators so as to do the best it can, given what it knows. A diagnostic system that reads a retinal photograph and outputs a probability of diabetic retinopathy is a rational agent in exactly this sense: its "sensor" is the image, its "actuator" is the report, and "doing well" is defined by a measurable objective (agreement with ophthalmologists). This definition has two virtues for engineers: it does not require the system to think like a person, and it makes the objective explicit, so success can be measured.

### 1.2 Four words that are often confused: system, algorithm, model, equation

The lecture insists on vocabulary because loose use of these words is the source of much confusion in the literature and the press.

- A **computer system** is the combination of hardware, software and peripheral devices that performs a task. "The AI" that reads a radiograph is a system: a GPU, an operating system, a program, a network connection to the hospital's image archive.
- An **algorithm** is a finite, unambiguous sequence of steps that transforms an input into an output. Needleman&ndash;Wunsch (BGM324), gradient descent (Unit 2) and k-nearest neighbours (Unit 3) are algorithms. An algorithm has no parameters learned from data; it is a procedure.
- A **model** is an object with adjustable parameters that maps an input to an output. A fitted logistic regression, a trained decision tree and a neural network with its weights are models. A *learning algorithm* produces a model from data; the model, not the algorithm, is what is deployed. When the news says "the algorithm decided", it almost always means "the model predicted".
- An **equation** is a mathematical statement of a relationship; a *mathematical model* is one or more equations taken as a description of a phenomenon. The nearest-centroid rule of Section 5 is an equation; with its two centroids filled in from data, it becomes a model.

### 1.3 What a system must do to be called intelligent

Operationally, the lecture lists six capacities that AI systems exhibit to some degree, and the list is a useful checklist for reading any claim about an "intelligent" system: **reasoning and problem-solving** (analysing data, identifying patterns, drawing conclusions); **learning** (improving performance from data and experience &mdash; the subject of Units 2&ndash;8); **perception** (interpreting visual, acoustic or linguistic input &mdash; computer vision, speech recognition, natural language processing); **decision-making** (choosing an action from inputs, rules or learned patterns, usually by optimising an objective); **automation** (performing the task without a human in the loop); and **adaptability** (changing behaviour when the environment changes). A spam filter has learning, perception of text and decision-making but no adaptability unless retrained; a chess program has reasoning and decision-making but no perception; AlphaFold has learning and perception of sequences. No present system has all six in the general sense a person does, which is the content of the next distinction.

### 1.4 Narrow, general and super-intelligence; the four functional types

The systems that exist today are instances of <strong>narrow (weak) AI</strong>: each solves one task &mdash; classify images, transcribe speech, predict protein structure, play Go &mdash; and has no competence outside it. <strong>Artificial general intelligence (AGI)</strong>, a system with human-level competence across tasks, does not exist, and whether current methods lead to it is an open scientific and philosophical question that this course does not need to settle. When a news article says "an AI has learned to diagnose skin cancer", the subject is a narrow system trained for one classification problem, and the correct question to ask is the one this course teaches: <em>on which data, measured how, and does it still work on patients unlike those it was trained on?</em>

The word <em>intelligence</em> is also used for several distinct human capacities. Psychology distinguishes, among others, <strong>fluid</strong> intelligence (reasoning in novel situations) from <strong>crystallised</strong> intelligence (accumulated knowledge), and Gardner's influential though contested theory lists linguistic, logical&ndash;mathematical, spatial, musical, bodily&ndash;kinaesthetic, interpersonal, intrapersonal and naturalistic intelligences. The mapping to machines is instructive: current systems are very strong on crystallised, logical&ndash;mathematical and increasingly on linguistic and spatial tasks, and essentially absent on the interpersonal, intrapersonal and bodily dimensions. A system that answers medical examination questions above the passing threshold<sup class="c"><a href="#r4">[4]</a></sup> and a system that cannot pick a pen off a table are the same kind of thing.

Popular writing adds a third tier, **artificial super-intelligence (ASI)**, a hypothetical system that exceeds human capability in every domain; it is a subject of philosophy and policy, not of engineering, and this course mentions it only so that you recognise the term. A second, *functional* classification popularised by Hintze<sup class="c"><a href="#r30">[30]</a></sup> sorts systems by what they can do with the past:

| Type | What it does | Status | Example |
|---|---|---|---|
| **Reactive** | Responds to the present input with no memory of the past; its behaviour is fixed once built | Exists | IBM's Deep Blue (1997): evaluates the current board, remembers no previous game |
| **Limited memory** | Learns from stored past data and uses recent experience to decide; this is where every machine-learning system of this course sits | Exists | A trained classifier; a self-driving car tracking nearby vehicles over the last seconds |
| **Theory of mind** | Would model the beliefs, intentions and emotions of the agents it interacts with | Conceptual; under research | &mdash; |
| **Self-aware** | Would possess a representation of its own internal state | Hypothetical | &mdash; |

The lecture asks you to think about the risks commonly associated with the last two tiers &mdash; unpredictable behaviour, rapid self-improvement ("intelligence explosion"), manipulation, the absence of empathy or of moral judgement. Section 7 takes a different, and for an engineer more urgent, position: the systems that *exist* (limited-memory systems) already produce harm through mechanisms that are fully understood &mdash; re-identification, shortcut learning, label bias, misinformation &mdash; and preventing those is a professional responsibility now, whatever one believes about the hypothetical tiers.

<div class="box warn"><h4>Common misconception</h4><p>"Artificial intelligence" and "machine learning" are not synonyms. Machine learning (Unit 2) is the sub-field of AI in which the behaviour of the system is <em>learned from data</em> rather than programmed. Much of the AI that ran hospitals in the 1980s &mdash; expert systems &mdash; contained no learning at all, and much of what is marketed as AI today is machine learning. The course title is "Artificial Intelligence" and its content is mostly machine learning, because that is where the field's practical impact has concentrated since the 2010s; Section 3 explains why.</p></div>

## 2. A short history, with its two winters

The history of the field is a cycle of enthusiasm, over-promise, disappointment and recovery. Knowing it protects you from both the hype and the backlash of the current cycle. The figure below places the main events on a timeline; the paragraphs that follow explain them.



In [ ]:
# Timeline of artificial intelligence: the events discussed in Section 2.
events <- data.frame(
  year = c(1943,1950,1956,1958,1966,1969,1976,1986,1997,2011,2012,2016,2017,2020,2022),
  lab  = c("McCulloch & Pitts\nartificial neuron","Turing, 'Computing\nMachinery and Intelligence'",
           "Logic Theorist;\nDartmouth: the name 'AI'","Rosenblatt,\nperceptron","ELIZA",
           "Minsky & Papert,\n'Perceptrons'","MYCIN (medical\nexpert system)",
           "Backpropagation\n(Rumelhart, Hinton,\nWilliams)","Deep Blue beats\nKasparov",
           "IBM Watson\nwins Jeopardy!","AlexNet wins\nImageNet","AlphaGo beats\nLee Sedol",
           "Transformer\narchitecture","AlphaFold2","ChatGPT"),
  lev  = c(1,-1,2,-2,-2,1,-1,1,-1,-2,1,-1,2,-2,1))
winters <- data.frame(a=c(1974,1987), b=c(1980,1993), lab=c("first AI winter","second AI winter"))
h <- c(`-2`=-1.25, `-1`=-0.5, `1`=0.5, `2`=1.25)
par(mar=c(3,1,3,1))
plot(NA, xlim=c(1938,2028), ylim=c(-2.1,2.15), axes=FALSE, xlab="", ylab="",
     main="Milestones of artificial intelligence, with the two 'winters' of reduced funding shaded")
for(i in 1:nrow(winters)){ rect(winters$a[i],-2.1,winters$b[i],2.15,col="#dfe7f1",border=NA)
  text((winters$a[i]+winters$b[i])/2,1.95,winters$lab[i],col="#2b5d9b",font=3,cex=.75) }
abline(h=0, lwd=1)
for(i in 1:nrow(events)){ hi <- h[as.character(events$lev[i])]
  segments(events$year[i],0,events$year[i],hi,col="#2b5d9b",lwd=.9)
  points(events$year[i],0,pch=19,col="#d2691e",cex=.9)
  text(events$year[i], hi+ifelse(hi>0,.07,-.07), events$lab[i],
       adj=c(.5,ifelse(hi>0,0,1)), cex=.62) }
axis(1, at=seq(1940,2020,10), pos=-2.1)



<p class="caption"><b>Figure 1.</b> A timeline of the events discussed in this section. The shaded bands are the two periods of reduced funding and public interest. Note the gap between the perceptron (1958) and the method that finally made multi-layer networks trainable (1986), and between that method and the hardware that made it practical at scale (2012).</p>

### 2.1 Foundations (1943&ndash;1956)

In 1943 the neurophysiologist Warren McCulloch and the logician Walter Pitts showed that a simplified model of a neuron &mdash; a unit that sums weighted inputs and fires if the sum exceeds a threshold &mdash; can compute any logical function when units are connected in networks.<sup class="c"><a href="#r5">[5]</a></sup> This is the ancestor of every artificial neural network (Unit 6). In 1949 the psychologist Donald Hebb proposed that the connection between two neurons strengthens when they fire together &mdash; <strong>Hebbian learning</strong>, the first rule for changing connection weights from experience, and the conceptual seed of every training algorithm.<sup class="c"><a href="#r31">[31]</a></sup> Turing's 1950 paper supplied the philosophical framing and, remarkably, anticipated machine learning: rather than programming an adult mind, he suggested, "why not rather try to produce one which simulates the child's?" and then educate it.<sup class="c"><a href="#r2">[2]</a></sup> In 1955&ndash;56 Allen Newell and Herbert Simon wrote the <strong>Logic Theorist</strong>, usually called the first artificial-intelligence program: it proved 38 of the first 52 theorems of Whitehead and Russell's <em>Principia Mathematica</em>, and found a shorter proof for one of them.<sup class="c"><a href="#r32">[32]</a></sup> The Dartmouth workshop of 1956 gave the field its name and its first generation of leaders: McCarthy, Minsky, Newell and Simon.<sup class="c"><a href="#r1">[1]</a></sup> The same years produced the first high-level programming languages &mdash; FORTRAN (1957), LISP (1958, by McCarthy, for AI) and COBOL (1959) &mdash; without which none of this could have been written.

### 2.2 The first wave and the first winter (1956&ndash;1980)

The early years produced programs that proved theorems, solved algebra word problems and played checkers, and generated predictions that proved badly wrong (Simon in 1965: machines would do any work a man can do within twenty years). Two artefacts of the period are still cited. <strong>ELIZA</strong> (Joseph Weizenbaum, 1966), the first chatbot, imitated a Rogerian psychotherapist by reflecting the user's sentences back as questions; it understood nothing, and Weizenbaum was alarmed by how readily users confided in it &mdash; the first documented case of people attributing understanding to a system that has none, a phenomenon now called the ELIZA effect.<sup class="c"><a href="#r33">[33]</a></sup> <strong>WABOT-1</strong> (Waseda University, 1972&ndash;73) was the first full-scale anthropomorphic robot, able to walk, grip and converse in Japanese after a fashion. Frank Rosenblatt's <strong>perceptron</strong> (1958) was the first learning machine: a McCulloch&ndash;Pitts neuron whose weights were adjusted automatically from labelled examples.<sup class="c"><a href="#r6">[6]</a></sup> In 1969 Minsky and Papert proved that a single-layer perceptron cannot represent functions as simple as exclusive-or, and argued that multi-layer versions would be untrainable.<sup class="c"><a href="#r7">[7]</a></sup> The argument was influential, funding for neural networks collapsed, and together with the 1973 Lighthill report in the United Kingdom, which concluded that AI had failed to meet its promises, it produced the <strong>first AI winter</strong>.

### 2.3 Expert systems, medicine, and the second winter (1970&ndash;1993)

The field recovered by narrowing its ambition. <strong>Expert systems</strong> encoded the knowledge of a human specialist as hundreds or thousands of IF&ndash;THEN rules and reasoned with them. Medicine was the showcase. <strong>MYCIN</strong>, developed at Stanford from 1972, recommended antibiotic therapy for bacterial infections from about 600 rules, and introduced <em>certainty factors</em> to reason under uncertainty &mdash; a rule could conclude "the organism is <em>Pseudomonas</em> with certainty 0.7".<sup class="c"><a href="#r8">[8]</a></sup> In a blinded evaluation its recommendations were rated acceptable at least as often as those of infectious-disease faculty. It was never used clinically &mdash; for legal, workflow and computing reasons &mdash; but it established that a machine could match specialists in a narrow domain, and the questions it raised (who is liable? how does the system explain itself?) are the questions the FDA and the European Union are regulating today. Section 3 revisits MYCIN's style of reasoning in code.

Expert systems became a billion-dollar industry in the 1980s &mdash; the first national conference of the American Association for Artificial Intelligence was held at Stanford in 1980, and <strong>XCON</strong> (R1), which configured VAX computer orders for Digital Equipment Corporation from some 2,500 rules, was reported to save the company tens of millions of dollars a year &mdash; and the industry collapsed by the early 1990s. The reason is instructive: the rules had to be written by hand, the <em>knowledge acquisition bottleneck</em>; they were brittle, failing abruptly outside the situations their authors had anticipated; and they could not improve with experience. The <strong>second AI winter</strong> followed.

### 2.4 Learning from data (1986&ndash;2011)

Two developments laid the ground for the present. In 1986 Rumelhart, Hinton and Williams popularised <strong>backpropagation</strong>, the algorithm that trains multi-layer neural networks by propagating the output error backwards through the layers &mdash; the thing Minsky and Papert had considered infeasible.<sup class="c"><a href="#r9">[9]</a></sup> In parallel, statisticians and computer scientists built the <em>statistical</em> theory of learning from data: decision trees (Unit 4), support vector machines and Bayesian methods (Unit 3), ensemble methods, and the rigorous methodology of training and test sets (Supplement S1). IBM's Deep Blue defeating the world chess champion in 1997 was a milestone of search and engineering rather than of learning, but it changed public perception. The 2000s brought AI into homes and businesses quietly: the Roomba vacuum robot (2002) was the first mass-market autonomous robot, and by the middle of the decade recommendation systems at Netflix, Amazon and the social networks were machine-learning models serving hundreds of millions of users.

### 2.5 Deep learning (2012&ndash;present)

In 2012 a convolutional neural network, AlexNet, trained on graphics processing units (GPUs), reduced the error on the ImageNet image-recognition benchmark from 26&nbsp;% to 15&nbsp;%, a margin that no previous method had approached.<sup class="c"><a href="#r10">[10]</a></sup> The ingredients were old &mdash; the network architecture dates from 1989 &mdash; and the change was quantitative: a million labelled images and a thousand-fold increase in compute. Everything since follows the same recipe. Several events of the decade belong to the history of public perception rather than of method: IBM's Watson won the quiz show <em>Jeopardy!</em> in 2011 against the two best human champions, by combining hundreds of natural-language and retrieval components;<sup class="c"><a href="#r34">[34]</a></sup> in 2014 a chatbot persona, "Eugene Goostman", was declared to have passed a Turing test at the Royal Society by convincing 33&nbsp;% of judges in five-minute conversations that it was a 13-year-old Ukrainian boy &mdash; a result most researchers regard as showing the weakness of the test rather than the strength of the program; and in 2018 IBM's Project Debater held a live debate against champion debaters. AlphaGo defeated a world champion of Go in 2016 by combining deep networks with tree search and reinforcement learning;<sup class="c"><a href="#r11">[11]</a></sup> the Transformer architecture of 2017 made it possible to train language models on the whole web; AlphaFold2 solved the fifty-year-old protein-structure-prediction problem in 2020 to near-experimental accuracy,<sup class="c"><a href="#r12">[12]</a></sup> a result you studied in the bioinformatics course; and large language models have, since 2022, reached the passing threshold of medical licensing examinations.<sup class="c"><a href="#r4">[4]</a></sup>

### 2.6 Generative AI

The systems that brought AI to public attention after 2022 belong to a sub-family, <strong>generative AI</strong>: models trained on very large collections of existing content that produce <em>new</em> content of the same kind. Three modalities are in common use.

**Text to text.** Large language models are neural networks trained to predict the next word over corpora of trillions of words; the Transformer architecture made that training parallelisable at scale.<sup class="c"><a href="#r35">[35]</a></sup> The lecture names GPT (Generative Pre-trained Transformer, OpenAI) and BERT (Google). The two are different kinds of object and the distinction is worth keeping: GPT models are generative &mdash; they produce text &mdash; whereas BERT is an <em>encoder</em> trained to produce representations of text for classification and search, and does not generate.<sup class="c"><a href="#r36">[36]</a></sup> (The slides list BERT among "generative" models; this is the one correction this unit makes to them &mdash; BERT is an encoder.) Applications: assistants, translation, summarisation, code generation; in medicine, drafting of notes and the question-answering results cited above.<sup class="c"><a href="#r4">[4]</a></sup>

**Text to image, and image to video.** Diffusion models produce photorealistic images from a text prompt. The lecture recalls two cases that forced the question of authorship: in 2022 an image generated with Midjourney won the digital-art category of the Colorado State Fair, and in 2023 the photographer Boris Eldagsen won a Sony World Photography Award with an AI-generated image and refused the prize, saying he had entered it to test whether competitions were prepared for such entries. Video generation (Meta's Emu Video, 2023; OpenAI's Sora, 2024) extends the same models to moving images. Real-time video processing &mdash; face re-synthesis, gaze correction, speech recognition and machine translation in a video call (NVIDIA Maxine is the lecture's example) &mdash; uses the same family of methods.

**Why it matters for this course.** Generative models are built from the same components as the classifiers you will study &mdash; representation, loss function, optimisation, training sample &mdash; and they inherit the same failure modes: they reproduce the biases of their training data, they fail under distribution shift, and they are evaluated on benchmarks that may not reflect use. They add one failure mode of their own, the production of fluent, confident and false content, and one societal problem, treated in Section 7.6: content that cannot be told from the genuine article.

<div class="box key"><h4>What the history teaches</h4>
<ul>
<li>Every winter followed a period in which capability was over-stated and brittleness under-stated. The honest reporting of what a system <em>cannot</em> do is a professional skill, not pessimism.</li>
<li>The decisive shift was from <strong>programming knowledge</strong> (expert systems) to <strong>learning it from data</strong>. That shift is why data &mdash; its quantity, quality, representativeness and governance &mdash; is now the central engineering concern, and why Sections 4&ndash;7 are about data.</li>
<li>Progress has been driven as much by compute and data as by ideas: the neuron of 1943, the perceptron of 1958 and the backpropagation of 1986 are all present in a 2024 network.</li>
</ul></div>

## 3. The approaches to artificial intelligence

The course description speaks of "methods of knowledge formalisation, reasoning with and without uncertainty, and machine-learning techniques". These phrases name the main approaches of the field, which coexist rather than succeed one another.

### 3.1 Symbolic (knowledge-based) AI

The symbolic approach represents knowledge explicitly, as logical statements or rules, and derives conclusions by inference. Its tools are propositional and first-order logic, production rules, ontologies and knowledge graphs. Reasoning without uncertainty is classical logic; reasoning <em>with</em> uncertainty attaches degrees of belief to statements, as MYCIN's certainty factors did, or, in the modern formulation, uses probability theory and Bayesian networks (Unit 3). Symbolic systems are transparent &mdash; every conclusion has a traceable chain of rules &mdash; and brittle. In medicine they survive in clinical decision-support rules (drug&ndash;drug interaction alerts, dosing calculators) and in terminologies such as SNOMED CT and the Gene Ontology you met in bioinformatics.

### 3.2 Connectionist AI

The connectionist approach represents knowledge implicitly, distributed over the weights of a network of simple units, and acquires it by training. Nothing in a trained network corresponds to a readable rule. The approach is powerful where knowledge is hard to articulate &mdash; recognising a melanoma, transcribing speech &mdash; and opaque where an explanation is required. Unit 6 and Unit 8 are devoted to it.

### 3.3 Statistical (probabilistic) AI and machine learning

The statistical approach treats learning as estimation: given a sample of data, fit a model that predicts well on new data from the same source, and quantify the uncertainty of the prediction. Decision trees, k-nearest neighbours, support vector machines, naive Bayes, regression and clustering (Units 2&ndash;5) are statistical learning methods. The connectionist approach is, in modern practice, also statistical &mdash; a neural network is a very flexible statistical model fitted by optimisation &mdash; and the term <em>machine learning</em> covers both.

### 3.4 Evolutionary AI

The evolutionary approach treats problem-solving as search by simulated natural selection: a population of candidate solutions is evaluated, the fittest are selected, recombined and mutated, and the cycle repeats (Unit 7). It is used for optimisation problems with rugged objective landscapes &mdash; scheduling, design, and in your field the design of molecules with target properties.

### 3.5 Rules versus learned rules, in code

The shift from the symbolic to the statistical approach can be shown in twenty lines. The dataset is the one that recurs throughout this course: the <strong>Wisconsin breast-cancer</strong> data. The <code>mlbench</code> package<sup class="c"><a href="#r38">[38]</a></sup> ships the original Wolberg cytology version &mdash; 699 fine-needle aspirates (683 once the 16 records with a missing value are removed), each described by nine cytological attributes scored by a pathologist from 1 to 10 (clump thickness, uniformity of cell size, uniformity of cell shape, marginal adhesion, single epithelial cell size, bare nuclei, bland chromatin, normal nucleoli and mitoses), and labelled benign or malignant.<sup class="c"><a href="#r13">[13]</a></sup>

<div class="box warn"><h4>A note on the dataset (difference from the Python edition)</h4><p>scikit-learn's <code>load_breast_cancer</code> ships the later <em>diagnostic</em> Wisconsin set: 569 aspirates and 30 real-valued morphometric features (radius, texture, perimeter&nbsp;&hellip;). R's <code>mlbench::BreastCancer</code> is the earlier <em>original</em> Wisconsin set: more cases but nine integer-scored cytological attributes. The pedagogy is identical &mdash; the same hand rule versus learned rule, the same nearest-centroid classifier &mdash; but the feature names, thresholds and exact accuracies differ from the Python notebook because the two datasets differ. Every number below is computed from the R dataset in front of you.</p></div>



In [ ]:
# Load and curate the Wisconsin breast-cancer data (the recurring dataset of the course).
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                             # drop the 16 rows with a missing value
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal factor -> numeric 1..10
cat("rows x features:", nrow(bc), "x", ncol(bc) - 1, "\n")
table(bc$Class)



The curated table has 683 complete aspirates described by nine features; the class counts (444 benign, 239 malignant) show a mild imbalance of roughly 1&nbsp;:&nbsp;1.9. Before writing any rule, the next chunk looks at two of the nine features to see whether the classes separate at all.



In [ ]:
# Do malignant and benign aspirates occupy different regions of two features?
cols <- ifelse(bc$Class == "malignant", "#d2691e", "#2b5d9b")
plot(jitter(bc$Cell.size), jitter(bc$Bare.nuclei), col = cols, pch = 19, cex = .5,
     xlab = "uniformity of cell size (1-10)", ylab = "bare nuclei (1-10)",
     main = "Two cytological features (malignant = orange)")
legend("topleft", c("benign", "malignant"), col = c("#2b5d9b", "#d2691e"), pch = 19, bty = "n", cex = .9)



<p class="caption"><b>Figure 2.</b> Two of the nine cytological scores, jittered so that overlapping integer values are visible. Benign aspirates (blue) cluster at low cell size and few bare nuclei; malignant ones (orange) spread to high values of both. The classes occupy different regions &mdash; the separability that makes a rule possible.</p>

The first chunk writes a classification rule the way a 1980s knowledge engineer would &mdash; by asking a pathologist which features matter and choosing thresholds by hand. The second lets an algorithm find the rule from the data.



In [ ]:
# A hand-written, expert-system style rule: large, irregular nuclei are suspicious.
is_malignant <- bc$Class == "malignant"
rule_says_malignant <- (bc$Cell.size > 3) & (bc$Bare.nuclei > 3)   # thresholds chosen by hand
accuracy <- mean(rule_says_malignant == is_malignant)
cat(sprintf("Cases: %d  (malignant %d, benign %d)\n", nrow(bc), sum(is_malignant), sum(!is_malignant)))
cat(sprintf("Hand-written rule agrees with the biopsy in %.1f%% of cases\n", 100 * accuracy))



The hand rule &mdash; "cell size above 3 AND bare nuclei above 3 means malignant" &mdash; agrees with the biopsy in 89.0&nbsp;% of the 683 cases. That is already useful, and it was written without any fitting. The next chunk lets the data choose the features and thresholds.



In [ ]:
# The same task, but the rule is LEARNED from the data (a depth-2 decision tree; Unit 4 opens this box).
lab <- factor(ifelse(is_malignant, "malignant", "benign"))
tree <- rpart(lab ~ ., data = data.frame(bc[, 1:9], lab = lab), method = "class",
              control = rpart.control(maxdepth = 2, cp = 0, xval = 0))   # rpart = R's CART
print(tree)
cat(sprintf("\nLearned rule agrees with the biopsy in %.1f%% of cases (on the data it was fitted to)\n",
            100 * mean(predict(tree, type = "class") == lab)))



The learned tree<sup class="c"><a href="#r40">[40]</a></sup> splits first on cell size, then on bare nuclei and cell shape, and agrees with the biopsy in 95.5&nbsp;% of cases. The chunk below draws it.



In [ ]:
rpart.plot(tree, type = 2, extra = 2, box.palette = list("#dfe7f1", "#fbe3d3"),
           main = "A depth-2 decision tree learned from the data")



<p class="caption"><b>Figure 3.</b> The learned rule drawn as a tree. Each internal node is an IF&ndash;THEN test on one feature; each leaf gives the predicted class and the count of cases it covers. The rule has the same readable IF&ndash;THEN shape as the hand-written one, but its features and thresholds were chosen by an algorithm that examined all 683 cases and all nine features.</p>

The learned rule has the same IF&ndash;THEN shape as the hand-written one &mdash; it is still symbolic and readable &mdash; but its features and thresholds were chosen by an algorithm, and it agrees with the biopsy more often. Two caveats introduce the two methodological themes of the course. First, the learned rule's accuracy was measured on the same cases it was fitted to, which flatters it: Supplement S1 explains why and how to measure honestly. Second, the <em>cost</em> of the two kinds of error is not the same &mdash; a malignant tumour called benign is a missed cancer; a benign one called malignant is an unnecessary biopsy &mdash; and "accuracy" treats them as equal. Keep both caveats in mind; they recur in every unit.

<div class="box key"><h4>Exercise 1</h4><p>Change the thresholds of the hand-written rule (try cell size 2 and bare nuclei 2, then cell size 4 and 5) and observe how the agreement changes. Then change <code>maxdepth</code> of the tree to 1 and to 4. Why does a deeper tree agree more with the biopsy on these data, and why should that make you suspicious rather than pleased?</p></div>

## 4. Pattern recognition

### 4.1 Definition

<strong>Pattern recognition</strong> is the scientific discipline whose goal is the classification of objects into categories or classes, where the objects may be images, signals, measurements or any other kind of observation.<sup class="c"><a href="#r14">[14]</a></sup> The word <em>pattern</em> means a regularity in data that is informative about a category: the particular combination of nuclear size, shape and texture that distinguishes a malignant from a benign cell is a pattern; the sequence of voltage deflections that distinguishes atrial fibrillation from sinus rhythm in an electrocardiogram is a pattern. The discipline predates the term "machine learning" &mdash; its textbooks date from the 1970s &mdash; and today the two overlap almost completely: machine learning supplies the methods, pattern recognition supplies the problem formulation.

### 4.2 The pipeline

Every pattern-recognition system, from a 1970s optical character reader to a 2024 radiology model, has the same stages:

1. **Sensing** &mdash; a physical measurement produces raw data: a camera produces pixel intensities, an electrode produces voltages, a mass spectrometer produces intensities per mass-to-charge ratio, a questionnaire produces answers.
2. **Pre-processing and segmentation** &mdash; noise is removed, the object of interest is isolated: the nucleus is segmented from the background, the heartbeat is cut from the continuous recording.
3. **Feature extraction** &mdash; the object is described by a fixed-length vector of numbers, the <strong>feature vector</strong>: radius, texture, perimeter&nbsp;&hellip; for the nucleus. This is the step that turns a thing into mathematics, and the one where domain knowledge enters. In deep learning the features are learned rather than designed, but the vector is still there &mdash; it is the activation of the network's penultimate layer.
4. **Classification** &mdash; a function maps the feature vector to a class (or to a probability over classes). The function is the <em>classifier</em>; its parameters are set by learning from the training sample.
5. **Post-processing and decision** &mdash; the class output is combined with costs, context and other evidence to produce an action: biopsy or follow-up, alarm or no alarm.

This course concentrates on stages 3 and 4 and on how to evaluate the whole. Stage 1 is the domain of the instrumentation courses of your degree, and stage 5 is the domain of clinical practice and health policy; a bioengineer sits at the interface and must understand all five.

### 4.3 Representation: the feature vector and the data matrix

An object described by \( p \) measured features is a point in a \( p \)-dimensional space, written as a vector \( \mathbf{x} = (x_1, x_2, \dots, x_p) \). A sample of \( n \) objects is a matrix \( \mathbf{X} \) of \( n \) rows (objects, observations, instances, examples, cases &mdash; the terms are synonyms) and \( p \) columns (features, variables, attributes, descriptors &mdash; also synonyms):

\[ \mathbf{X} = \begin{pmatrix} x_{11} & x_{12} & \cdots & x_{1p} \\ x_{21} & x_{22} & \cdots & x_{2p} \\ \vdots & & \ddots & \vdots \\ x_{n1} & x_{n2} & \cdots & x_{np} \end{pmatrix}, \qquad \mathbf{y} = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix} \]

When each object also carries a known class \( y_i \), the pair \( (\mathbf{X}, \mathbf{y}) \) is a <strong>labelled</strong> dataset and the class vector \( \mathbf{y} \) is the <em>target</em>, <em>label</em> or <em>response</em>. The convention "rows are observations, columns are features" is universal in R's <code>data.frame</code>, in scikit-learn, in pandas and in the statistical literature; a dataset in which it is reversed must be transposed before any method is applied. In the breast-cancer data, \( n = 683 \) and \( p = 9 \):



In [ ]:
# The data matrix X (683 x 9) and the label vector y; the first rows of six features.
X <- bc[, 1:9]; y <- bc$Class
cat("dim(X) (n objects, p features):", dim(X)[1], "x", dim(X)[2], "| length(y):", length(y), "\n")
cat("class labels and counts:\n"); print(table(y))
head(X[, 1:6], 5)



Each row is one aspirate; each column is one cytological score from 1 to 10. The first patient has a clump thickness of 5 and a uniformity of cell size of 1 &mdash; low scores, consistent with the benign label. The dataset has 239 malignant and 444 benign cases, the mild imbalance noted above; in screening problems where disease prevalence is 1&nbsp;%, such imbalance becomes the dominant difficulty (Supplement S1).

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>You have already built data matrices without calling them that. A multiple sequence alignment is a matrix whose rows are sequences and whose columns are positions; a BLOSUM matrix is a feature of pairs of residues; the molecular descriptors of Unit 4 of BGM324 (molecular weight, cLogP, TPSA, rotatable bonds&nbsp;&hellip;) are a feature vector for a molecule, and a QSAR model is a pattern-recognition system whose objects are molecules and whose target is activity. The methods of this course apply to those matrices unchanged.</p></div>

### 4.4 Types of data

The feature type determines which mathematical operations are meaningful, and therefore which methods can be applied. Confusing types is the most common error in applied work.

| Type | Definition | Admissible operations | Clinical examples |
|---|---|---|---|
| **Continuous** (interval or ratio) | Real numbers; differences and (for ratio scales) ratios are meaningful | mean, standard deviation, distance, correlation | glucose (mg/dL), nuclear radius (µm), systolic pressure, age |
| **Discrete** (count) | Non-negative integers from counting | the same as continuous, with care for small counts | number of previous admissions, lymph nodes examined, CAG repeats |
| **Binary** (dichotomous) | Two states | proportion, agreement | sex at birth, smoker/non-smoker, event/no event |
| **Nominal** (unordered) | Labels with no order | mode, frequency; **no mean, no distance** | blood group, comuna of residence, ICD-10 code, nucleotide (A/C/G/T) |
| **Ordinal** (ordered) | Labels with an order but no fixed spacing | median, rank statistics; **no mean** in principle | tumour stage I&ndash;IV, pain 0&ndash;10, Glasgow Coma Scale |
| **Text, image, signal, sequence** | Structured raw data | must be converted by feature extraction | clinical notes, radiographs, ECG, DNA |

The important negative statements are in bold: a mean blood group does not exist, and the distance between blood groups A and B is undefined. Yet almost every method in this course computes distances or means. Categorical features must therefore be <em>encoded</em> as numbers in a way that does not invent an order: a nominal feature with \( k \) levels becomes \( k \) binary indicator columns (<strong>one-hot encoding</strong>); an ordinal feature may be mapped to its ranks if the analyst accepts that spacing is treated as equal. The next chunk builds a small clinical table that contains every type and shows the encodings. (The nine breast-cancer scores are themselves ordinal, which is why this course treats them as numbers only after noting that the equal-spacing assumption is being made.)



In [ ]:
# A synthetic clinical table containing every data type of Section 4.4.
set.seed(0); n <- 8
clinic <- data.frame(
  age_years        = sample(35:79, n, replace = TRUE),                               # continuous (ratio)
  glucose_mg_dL    = round(rnorm(n, 105, 20)),                                        # continuous
  prior_admissions = rpois(n, 1.2),                                                   # discrete count
  smoker           = sample(c(TRUE, FALSE), n, replace = TRUE),                       # binary
  blood_group      = sample(c("O", "A", "B", "AB"), n, replace = TRUE),              # nominal
  tumour_stage     = factor(sample(c("I","II","III","IV"), n, replace = TRUE),
                            levels = c("I","II","III","IV"), ordered = TRUE),          # ordinal
  stringsAsFactors = FALSE)
str(clinic)
clinic


In [ ]:
# Encodings that respect the type: one-hot for the nominal feature, rank codes for the ordinal one.
onehot <- model.matrix(~ blood_group - 1, clinic)                 # 4 indicator columns, no order
colnames(onehot) <- sub("blood_group", "bg_", colnames(onehot))
encoded <- data.frame(clinic[c("age_years","glucose_mg_dL","prior_admissions")],
                      smoker = as.integer(clinic$smoker),
                      onehot,
                      stage_code = as.integer(clinic$tumour_stage))  # ordinal: 1,2,3,4 preserves order
cat("From", ncol(clinic), "clinical variables to", ncol(encoded), "numerical features\n")
head(encoded)



Note what happened to <code>blood_group</code>: one column with four labels became four columns of zeros and ones, so that the Euclidean distance between any two different blood groups is the same (\( \sqrt{2} \)) and no group is "between" two others. Had we coded O&nbsp;=&nbsp;1, A&nbsp;=&nbsp;2, B&nbsp;=&nbsp;3, AB&nbsp;=&nbsp;4, every distance-based method would have treated AB as three steps further from O than A is, a statement with no biological meaning. The ordinal <code>tumour_stage</code>, by contrast, is defensibly coded 1&ndash;4, because the order is real; the analyst accepts that the steps are treated as equal. Converting free text to features &mdash; by counting words, by dictionaries of clinical terms, or by the embeddings of a language model &mdash; is a discipline of its own (natural language processing) that Unit 9 touches on.

### 4.5 Images and signals are feature vectors too

An image is a grid of pixel intensities; flattening the grid row by row gives a vector whose length is the number of pixels. A one-second electrocardiogram sampled at 500&nbsp;Hz is already a vector of 500 numbers, or can be summarised by a handful of designed features (heart rate, QRS duration, spectral power in a band). The next two chunks make the point. Because R has no built-in handwritten-digit dataset, we <em>simulate</em> small 8&nbsp;&times;&nbsp;8 images &mdash; a stand-in for the classical digits benchmark<sup class="c"><a href="#r15">[15]</a></sup> that is small enough to inspect by eye &mdash; and a synthetic physiological signal. Each simulated class is a template stroke plus noise.



In [ ]:
# Simulate 8x8 "stroke" images of four classes; an image is a point in a 64-dimensional space.
set.seed(0)
templates <- list(
  vbar  = {m <- matrix(0,8,8); m[,4:5] <- 13; m},            # vertical stroke
  hbar  = {m <- matrix(0,8,8); m[4:5,] <- 13; m},            # horizontal stroke
  cross = {m <- matrix(0,8,8); m[,4:5] <- 13; m[4:5,] <- 13; m},  # plus sign
  diag  = {m <- matrix(0,8,8); m[cbind(1:8,1:8)] <- 13; m[cbind(1:8,pmin(1:8+1,8))] <- 13; m})
classes <- names(templates)
make_sample <- function(tpl) pmax(0, pmin(16, round(tpl + rnorm(64, 0, 2))))
Ximg <- do.call(rbind, lapply(rep(classes, each = 40), function(k) as.vector(make_sample(templates[[k]]))))
yimg <- factor(rep(classes, each = 40), levels = classes)
cat("Image shape: 8 x 8 -> vector length:", ncol(Ximg), "| data matrix X:", nrow(Ximg), "x", ncol(Ximg), "\n")

show_img <- function(v, main="") image(t(matrix(v,8,8)[8:1,]), col = gray(seq(1,0,length=17)),
                                       zlim = c(0,16), axes = FALSE, main = main)
layout(matrix(c(1,2,2,2), 1)); par(mar = c(3,2,2,1))
show_img(Ximg[1,], main = "class 'vbar' as an 8 x 8 image")
barplot(Ximg[1,], col = "#2b5d9b", border = NA, space = 0,
        main = "the same object as a 64-element feature vector (pixels, row by row)",
        xlab = "vector index", ylab = "intensity (0-16)")


In [ ]:
# The 'average image' of each class: the class pattern (template) in pixel space.
par(mfrow = c(1, length(classes)), mar = c(1,1,2,1))
for (k in classes) show_img(colMeans(Ximg[yimg == k, ]), main = k)



<p class="caption"><b>Figure 4.</b> Top: one simulated "vbar" image shown as an 8&nbsp;&times;&nbsp;8 picture and as the 64-element vector that a classifier actually sees. Bottom: the mean of all 40 images of each class. Averaging over a class produces a clean template of the pattern &mdash; the regularity that all members of a class share and that distinguishes them from the others. The simplest classifier of Section 5 does exactly this: it compares a new object to each class template.</p>



In [ ]:
# A signal as a pattern: synthetic 'heartbeats' from two rhythms, and a two-feature summary of each.
set.seed(1); fs <- 250; seconds <- 4; t <- seq(0, seconds, by = 1/fs)
beat_train <- function(rate_bpm, jitter, n_seg) t(sapply(1:n_seg, function(i) {
  peaks <- cumsum(rnorm(20, 60/rate_bpm, jitter)); peaks <- peaks[peaks < seconds]
  rowSums(sapply(peaks, function(pk) exp(-((t - pk)/0.02)^2))) + rnorm(length(t), 0, 0.05) }))
regular <- beat_train(70, 0.01, 30); irregular <- beat_train(95, 0.15, 30)
find_peaks <- function(x, h, d) { c <- which(x[2:(length(x)-1)] > x[1:(length(x)-2)] &
  x[2:(length(x)-1)] >= x[3:length(x)] & x[2:(length(x)-1)] > h) + 1
  keep <- integer(0); last <- -Inf; for (i in c) if (i - last >= d) { keep <- c(keep, i); last <- i }; keep }
features <- function(sig) { idx <- find_peaks(sig, 0.5, round(0.25*fs)); rr <- diff(t[idx]); c(mean(rr), sd(rr)) }
F <- t(apply(rbind(regular, irregular), 1, features)); labs <- rep(0:1, each = 30)
par(mfrow = c(1, 2), mar = c(4,4,2,1))
plot(t, regular[1,] + 1.6, type = "l", col = "#2b5d9b", lwd = .8, xlab = "time (s)", ylab = "",
     ylim = range(c(regular[1,] + 1.6, irregular[1,])), yaxt = "n", main = "raw signal: 1000 samples / segment")
lines(t, irregular[1,], col = "#d2691e", lwd = .8)
legend("topright", c("regular (70 bpm)","irregular (95 bpm)"), col = c("#2b5d9b","#d2691e"), lwd = 1, bty = "n", cex = .8)
plot(F[labs==0,1], F[labs==0,2], col = "#2b5d9b", pch = 19, cex = .7, xlab = "mean beat interval (s)",
     ylab = "interval variability (s)", xlim = range(F[,1]), ylim = range(F[,2]),
     main = "feature space: 2 numbers / segment")
points(F[labs==1,1], F[labs==1,2], col = "#d2691e", pch = 19, cex = .7)



<p class="caption"><b>Figure 5.</b> Left: one segment of each synthetic rhythm, 1,000 samples long. Right: the same 60 segments after feature extraction &mdash; each segment is now a point described by two numbers, the mean interval between beats and its variability &mdash; and the two rhythms occupy separate regions. Feature extraction reduced the dimension from 1,000 to 2 while keeping the information that matters for the task. Whether it keeps enough is an empirical question answered by the classifier's performance, never by inspection alone.</p>

<div class="box key"><h4>Key idea</h4><p>A pattern is a <em>region of feature space</em> occupied preferentially by one class. A classifier is a rule that partitions feature space into regions and assigns a class to each. The art of pattern recognition is to find a representation in which the classes occupy separable regions, and a rule that finds the boundaries from a finite sample without fitting the sample's accidents.</p></div>

### 4.6 The training sample

The <strong>training sample</strong> (training set) is the labelled dataset from which the classifier's parameters are learned. Three properties of it determine everything that follows in the course.

**It is a sample, not the population.** The Wisconsin aspirates were collected at one hospital, imaged with one protocol, and labelled by one pathology service. The classifier will be asked to classify aspirates from other hospitals, microscopes and populations. Learning is <em>inductive</em>: it generalises from the cases seen to the cases not seen, and that generalisation is only justified if the unseen cases come from the same distribution as the seen ones. When they do not &mdash; a different scanner, an older population, a different disease prevalence &mdash; the situation is called <strong>dataset shift</strong>, and performance degrades, often silently. Section 7.3 shows a mechanism.

**Its labels define the task.** The classifier learns to reproduce whatever the label is, not what the analyst intends the label to mean. If the label is "biopsy result", the system learns to predict biopsies; if the label is "healthcare cost", the system learns to predict cost, even if the analyst calls it "health need". Section 7.4 shows how far these can diverge.

**Its size bounds what can be learned.** With \( n \) examples and \( p \) features, a flexible model can memorise the sample without learning the pattern. The ratio \( n/p \), the complexity of the model and the noise in the labels together determine how much of what the model learns is pattern and how much is accident; the methodology of Supplement S1 exists to measure the difference. Section 5.3 shows the dependence on \( n \) with the simplest possible classifier.

## 5. The simplest classifier, by hand and in code

To make the pipeline concrete before the course introduces real algorithms, this section builds the simplest classifier there is: the <strong>nearest-centroid</strong> (minimum-distance) classifier. Its training consists of computing the mean feature vector &mdash; the <em>centroid</em> &mdash; of each class. Its decision rule assigns a new object to the class whose centroid is closest in Euclidean distance:

<div class="eq">\[ \hat{y}(\mathbf{x}) = \arg\min_{c} \; \lVert \mathbf{x} - \boldsymbol{\mu}_c \rVert_2, \qquad \boldsymbol{\mu}_c = \frac{1}{n_c} \sum_{i:\, y_i = c} \mathbf{x}_i \]
<span class="where">where \( \boldsymbol{\mu}_c \) is the centroid of class \( c \) computed from the \( n_c \) training objects of that class, and \( \lVert\cdot\rVert_2 \) is the Euclidean norm. The decision boundary between two classes is the perpendicular bisector of the segment joining their centroids &mdash; a straight line in two dimensions, a plane in three, a hyperplane in general.</span></div>

### 5.1 Worked example by hand

Take two illustrative features of an aspirate, \( x_1 \) and \( x_2 \) (on a common arbitrary scale), and six training cases, three of each class, with values rounded for hand calculation:

| case | \( x_1 \) | \( x_2 \) | class |
|---|---|---|---|
| 1 | 12 | 2 | benign |
| 2 | 13 | 3 | benign |
| 3 | 11 | 1 | benign |
| 4 | 19 | 9 | malignant |
| 5 | 21 | 11 | malignant |
| 6 | 17 | 7 | malignant |

**Training.** The centroids are the per-class means: \( \boldsymbol{\mu}_{B} = \left(\tfrac{12+13+11}{3}, \tfrac{2+3+1}{3}\right) = (12, 2) \) and \( \boldsymbol{\mu}_{M} = \left(\tfrac{19+21+17}{3}, \tfrac{9+11+7}{3}\right) = (19, 9) \).

**Classification** of a new case \( \mathbf{x} = (15, 4) \): \( d_B = \sqrt{(15-12)^2 + (4-2)^2} = \sqrt{9 + 4} = \sqrt{13} \approx 3.61 \) and \( d_M = \sqrt{(15-19)^2 + (4-9)^2} = \sqrt{16 + 25} = \sqrt{41} \approx 6.40 \). Since \( d_B < d_M \), the case is classified <strong>benign</strong>. The chunk below confirms the arithmetic and draws the decision boundary.



In [ ]:
# Nearest-centroid classifier on the six hand-worked cases, and its decision boundary.
train <- rbind(c(12,2), c(13,3), c(11,1), c(19,9), c(21,11), c(17,7)); cls <- c("B","B","B","M","M","M")
mu_B <- colMeans(train[cls=="B",]); mu_M <- colMeans(train[cls=="M",])
new <- c(15, 4); d_B <- sqrt(sum((new-mu_B)^2)); d_M <- sqrt(sum((new-mu_M)^2))
cat(sprintf("centroid benign = (%g, %g), centroid malignant = (%g, %g)\n", mu_B[1],mu_B[2],mu_M[1],mu_M[2]))
cat(sprintf("distance to benign = %.2f, to malignant = %.2f  ->  predicted: %s\n",
            d_B, d_M, ifelse(d_B < d_M, "benign", "malignant")))
plot(train, col = ifelse(cls=="B","#2b5d9b","#d2691e"), pch = 19, cex = 1.4, xlim = c(9,23), ylim = c(-1,13),
     xlab = "feature x1", ylab = "feature x2", main = "Nearest-centroid classifier on six training cases")
points(mu_B[1], mu_B[2], pch = 4, lwd = 3, cex = 2, col = "#2b5d9b")
points(mu_M[1], mu_M[2], pch = 4, lwd = 3, cex = 2, col = "#d2691e")
points(new[1], new[2], pch = 8, cex = 2, lwd = 2, col = "#3a7d44")
mid <- (mu_B + mu_M)/2; dir <- mu_M - mu_B; perp <- c(-dir[2], dir[1])
pts <- rbind(mid - 1.2*perp, mid + 1.2*perp); lines(pts, lty = 2)
legend("topleft", c("benign","malignant","centroids (x)","new case (8)","boundary"),
       col = c("#2b5d9b","#d2691e","black","#3a7d44","black"),
       pch = c(19,19,4,8,NA), lty = c(NA,NA,NA,NA,2), bg = "white", box.col = NA, cex = .8)



<p class="caption"><b>Figure 6.</b> The six training cases, the two centroids (crosses), the new case (star) and the decision boundary, which is the perpendicular bisector of the segment joining the centroids. Everything on the lower-left side is classified benign, everything on the upper-right malignant. Training a nearest-centroid classifier means computing two means; the boundary follows.</p>

### 5.2 The same classifier on the full dataset

The hand example used six cases chosen to be clean. On the real sample of 683, the classes overlap &mdash; some malignant tumours have low scores, some benign ones high &mdash; and no straight line separates them perfectly. The chunk trains the nearest-centroid classifier on a random 70&nbsp;% of the cases and tests it on the remaining 30&nbsp;%, which it never saw. The reason for holding cases out is the subject of Supplement S1; here the point is to see a real classifier work and to read its result correctly.



In [ ]:
# Nearest centroid on two real features, trained on 70% of the cases, tested on the other 30%.
set.seed(0)
feat2 <- c("Cl.thickness", "Cell.size")
X2 <- as.matrix(bc[, feat2]); yc <- bc$Class
idx <- sample(nrow(bc), round(0.7 * nrow(bc)))
cent <- t(sapply(levels(yc), function(c) colMeans(X2[idx, ][yc[idx] == c, ])))
euclid <- function(M, v) sqrt(rowSums(sweep(M, 2, v)^2))
predict_nc <- function(M) levels(yc)[apply(cbind(euclid(M, cent["benign",]), euclid(M, cent["malignant",])), 1, which.min)]
pred <- predict_nc(X2[-idx, ])
cat("centroids (rows = classes):\n"); print(round(cent, 2))
cat(sprintf("accuracy on the %d held-out cases: %.3f\n", nrow(bc) - length(idx),
            mean(pred == as.character(yc[-idx]))))
# decision regions over the plane
gx <- seq(0.5, 10.5, length = 220); gy <- seq(0.5, 10.5, length = 220)
grid <- expand.grid(Cl.thickness = gx, Cell.size = gy)
Z <- matrix(predict_nc(as.matrix(grid)) == "malignant", length(gx))
image(gx, gy, Z, col = c("#dfe7f1", "#fbe3d3"), xlab = "clump thickness (1-10)", ylab = "cell size (1-10)",
      main = "Nearest-centroid decision regions (205 test cases)")
te <- X2[-idx, ]; tc <- yc[-idx]
points(jitter(te[tc=="benign",1]), jitter(te[tc=="benign",2]), col = "#2b5d9b", pch = 19, cex = .5)
points(jitter(te[tc=="malignant",1]), jitter(te[tc=="malignant",2]), col = "#d2691e", pch = 19, cex = .5)
points(cent[,1], cent[,2], pch = 4, lwd = 3, cex = 2, col = c("#2b5d9b","#d2691e"))



<p class="caption"><b>Figure 7.</b> The two decision regions and the 205 test cases (jittered). Most benign cases (blue) fall in the blue region and most malignant cases (orange) in the orange region, but a band of cases near the boundary is misclassified in both directions. The accuracy printed above the figure &mdash; about 0.93 &mdash; is the fraction of test cases on the correct side. Two features and two means already recover more than nine in ten biopsy results; the rest of the course is about the remaining cases, and about knowing <em>which</em> ones.</p>



In [ ]:
# Exercise 2 in code: does using all nine features help this classifier?
cent9 <- t(sapply(levels(yc), function(c) colMeans(as.matrix(X[idx, ])[yc[idx] == c, ])))
d9 <- sapply(levels(yc), function(c) euclid(as.matrix(X[-idx, ]), cent9[c, ]))
pred9 <- levels(yc)[apply(d9, 1, which.min)]
cat(sprintf("test accuracy with 2 features: 0.927   with all 9 features: %.3f\n",
            mean(pred9 == as.character(yc[-idx]))))



Using all nine cytological scores raises the held-out accuracy from 0.927 to 0.951. 

<div class="box key"><h4>Exercise 2</h4><p>Replace the two features with <code>"Marg.adhesion"</code> and <code>"Mitoses"</code> and re-run the classifier of Section 5.2; the accuracy falls, because those two features separate the classes less well. Note a difference from the Python edition: there, adding all 30 diagnostic features <em>hurt</em> the nearest-centroid classifier, because features such as "mean area" (ranging past 2,500) dominated the Euclidean distance. Here all nine scores share the 1&ndash;10 scale, so no feature dominates and more features help. Feature scaling (Supplement S0 and Unit 3) is what makes the two editions behave differently.</p></div>

### 5.3 Why the size of the training sample matters

Section 4.6 claimed that the training sample's size bounds what can be learned. The chunk below measures it: the nearest-centroid classifier is trained on samples of increasing size drawn at random from the 478 training cases, tested on the same 205 held-out cases, and the experiment is repeated 50 times at each size to average over the luck of the draw.



In [ ]:
# Learning curve: test accuracy as a function of the number of training cases (50 draws per size).
set.seed(0)
sizes <- c(4, 8, 16, 32, 64, 128, 256, length(idx)); ytr <- droplevels(yc[idx]); Xtr <- X2[idx, ]; Xte <- X2[-idx, ]; yte <- as.character(yc[-idx])
acc <- matrix(0, length(sizes), 50)
for (i in seq_along(sizes)) for (r in 1:50) {
  repeat { s <- sample(length(ytr), sizes[i]); if (length(unique(ytr[s])) == 2) break }
  ce <- t(sapply(levels(ytr), function(c) colMeans(Xtr[s, ][ytr[s] == c, , drop = FALSE])))
  pr <- levels(ytr)[apply(cbind(euclid(Xte, ce["benign",]), euclid(Xte, ce["malignant",])), 1, which.min)]
  acc[i, r] <- mean(pr == yte) }
summary_lc <- data.frame(n_train = sizes, mean_accuracy = round(rowMeans(acc), 3),
                         worst_draw = round(apply(acc, 1, min), 3), best_draw = round(apply(acc, 1, max), 3))
plot(sizes, rowMeans(acc), type = "n", log = "x", ylim = c(0.5, 1), xlab = "number of training cases",
     ylab = "accuracy on 205 test cases", main = "Learning curve of the nearest-centroid classifier")
polygon(c(sizes, rev(sizes)), c(apply(acc,1,quantile,.05), rev(apply(acc,1,quantile,.95))),
        col = "#dfe7f1", border = NA)
lines(sizes, rowMeans(acc), type = "b", pch = 19, col = "#2b5d9b")
legend("bottomright", c("mean test accuracy","5th-95th percentile of 50 draws"),
       col = c("#2b5d9b","#dfe7f1"), lwd = c(1,6), pch = c(19,NA), bty = "n", cex = .8)
summary_lc



<p class="caption"><b>Figure 8.</b> Mean test accuracy (line) and its 5th&ndash;95th percentile band over 50 random training samples of each size. With four training cases the classifier is unreliable &mdash; the worst draws fall well below the plateau &mdash; and the band is wide; by a few dozen cases the mean has plateaued near 0.93 and the band has narrowed. Two lessons: a small training sample gives a classifier whose quality depends on luck, and more data stops helping once the model's own limitations (here, a straight boundary) dominate. Richer models (Units 3, 4, 6) keep improving for longer, and need correspondingly more data.</p>

## 6. Applications in medicine and bioengineering

The applications of pattern recognition in health are too many to list; this section selects examples that illustrate the forms the problem takes, and that are documented in primary literature you can read. Each is a classification or prediction problem in the sense of Section 4.

**Medical imaging** is the field in which deep learning first matched specialists. A convolutional network trained on 128,175 retinal photographs detected referable diabetic retinopathy with a sensitivity of 97&nbsp;% and specificity of 93&nbsp;% in one validation set, on a par with a panel of ophthalmologists;<sup class="c"><a href="#r16">[16]</a></sup> a network trained on 129,450 clinical images classified skin lesions with dermatologist-level accuracy;<sup class="c"><a href="#r17">[17]</a></sup> and the first autonomous AI diagnostic system &mdash; one that renders a decision without a clinician reading the image &mdash; was authorised by the FDA in 2018 for diabetic retinopathy screening. By 2020 more than 60 AI-based medical devices had been approved by the FDA, the majority in radiology.<sup class="c"><a href="#r18">[18]</a></sup> Unit 8 is devoted to image classification.

**Physiological signals.** A network trained on 91,232 single-lead ECG recordings from 53,549 patients classified twelve rhythm classes with performance exceeding that of individual cardiologists on a test set with consensus labels.<sup class="c"><a href="#r19">[19]</a></sup> The single-lead setting matters: it is the one produced by wearable devices, and it moved rhythm classification from the hospital to the wrist.

**Clinical prediction from the electronic record.** Models that predict deterioration, sepsis or readmission from routinely recorded variables are now deployed in many hospitals. They are also where the failures of Section 7 have been documented most clearly: an external validation of a widely deployed proprietary sepsis model found a sensitivity of 33&nbsp;% and a positive predictive value of 12&nbsp;% at the vendor's recommended threshold, far below the reported development performance.<sup class="c"><a href="#r20">[20]</a></sup> The model was not wrong in its own data; the hospitals were different from the hospitals it was developed on.

**Molecular and structural biology** &mdash; the subject of the bioinformatics course. AlphaFold2 predicts protein structure from sequence with accuracy competitive with experiment;<sup class="c"><a href="#r12">[12]</a></sup> a neural network trained on 2,335 molecules' antibacterial activity, then applied to a library of 6,111 molecules in clinical development, identified halicin, a structurally novel antibiotic active against resistant pathogens;<sup class="c"><a href="#r21">[21]</a></sup> and the QSAR models you built in BGM324 are pattern-recognition systems whose objects are molecules.

**Clinical language.** Large language models answer medical questions at or above the passing level of licensing examinations,<sup class="c"><a href="#r4">[4]</a></sup> summarise records and draft notes. Their failure mode &mdash; fluent, confident, wrong statements &mdash; is new in kind, and their evaluation is an active research problem.

<div class="box key"><h4>Reading an application paper</h4><p>Workshop 2 of the course asks you to present a scientific article that applies supervised learning. The structure of Section 4 is the reading guide: identify the objects, the features, the labels and where they came from, the size and provenance of the training sample, the classifier, how performance was measured and on which cases (same hospital? same scanner? same years?), and what the costs of the two kinds of error are. A paper that does not answer these questions has not described a pattern-recognition system; it has advertised one.</p></div>

## 7. Data, privacy and ethics

The course description names "the importance of data protection from the standpoint of professional ethics" and "regulations and ethical problems related to the application of AI in health". These are engineering concerns, not addenda: in a discipline that learns from data about people, the handling of those data is part of the method. This section treats four problems &mdash; re-identification, dataset shift, label bias and the regulatory framework &mdash; and demonstrates the first three in code.

### 7.1 Personal data, sensitive data and de-identification

Health data are <strong>sensitive personal data</strong> in every modern data-protection regime. In Chile, Law 19.628 of 1999 on the protection of private life has governed personal data since 1999 and Law 20.584 of 2012 establishes the confidentiality of the clinical record; Law 21.719, published in December 2024 and scheduled to enter into force on 1 December 2026, rewrites the regime along the lines of the European General Data Protection Regulation (GDPR), creates a Personal Data Protection Agency with powers of inspection and sanction, and classifies health and genetic data as sensitive data requiring explicit consent or a specific legal basis.<sup class="c"><a href="#r22">[22]</a></sup> The GDPR (Regulation (EU) 2016/679) is the reference regime internationally, and the United States' HIPAA Privacy Rule (1996) defines the de-identification standard most medical datasets follow.

The standard remedy for using health data in research is <strong>de-identification</strong>: removing direct identifiers (name, identity number, address, telephone). Two terms must be kept apart. <strong>Anonymisation</strong> is irreversible: no one can link the record back to the person. <strong>Pseudonymisation</strong> replaces identifiers with a code and keeps the key elsewhere; it is reversible by whoever holds the key, and the GDPR treats pseudonymised data as still personal. Most "anonymised" clinical datasets are in fact pseudonymised, and many are re-identifiable.

### 7.2 Re-identification and k-anonymity

Removing names does not anonymise a record, because combinations of ordinary attributes &mdash; <strong>quasi-identifiers</strong> &mdash; are often unique. Latanya Sweeney showed in 2000 that 87&nbsp;% of the United States population was uniquely identified by the triple {5-digit postal code, date of birth, sex}, and demonstrated it by re-identifying the medical record of the governor of Massachusetts in a "de-identified" hospital dataset by linking it to the public voter roll.<sup class="c"><a href="#r23">[23]</a></sup> A 2019 study estimated that 99.98&nbsp;% of Americans would be correctly re-identified in any dataset using 15 demographic attributes.<sup class="c"><a href="#r24">[24]</a></sup> The chunk below reproduces the mechanism on a synthetic population and shows the standard countermeasure, <strong>k-anonymity</strong>: generalising the quasi-identifiers until every combination is shared by at least \( k \) records.<sup class="c"><a href="#r23">[23]</a></sup>



In [ ]:
# Re-identification risk: how many records are unique given a few quasi-identifiers?
set.seed(0); N <- 20000
pop <- data.frame(
  birth_date = as.Date("1940-01-01") + sample(0:(365*65), N, replace = TRUE),
  sex        = sample(c("F","M"), N, replace = TRUE),
  comuna     = sample(sprintf("comuna_%02d", 0:59), N, replace = TRUE),   # 60 districts
  diagnosis  = sample(c("hypertension","diabetes","asthma","depression","HIV"), N,
                      replace = TRUE, prob = c(.4,.3,.15,.1,.05)))
share_unique <- function(cols) {
  key <- do.call(paste, pop[cols]); tt <- table(key)
  c(unique_frac = sum(tt[key] == 1) / N, k_min = min(tt)) }
for (cols in list("sex", c("sex","comuna"), c("sex","comuna","birth_date"))) {
  r <- share_unique(cols)
  cat(sprintf("%-34s -> unique %5.1f%%   (smallest group k = %d)\n",
              paste(cols, collapse = "+"), 100 * r["unique_frac"], r["k_min"])) }


In [ ]:
# k-anonymity by generalisation: replace the exact birth date by the year, then by a 10-year band.
pop$birth_year  <- as.integer(format(pop$birth_date, "%Y"))
pop$birth_decade <- (pop$birth_year %/% 10) * 10
for (cols in list(c("sex","comuna","birth_year"), c("sex","comuna","birth_decade"))) {
  r <- share_unique(cols)
  cat(sprintf("%-34s -> unique %5.1f%%   (smallest group k = %d)\n",
              paste(cols, collapse = "+"), 100 * r["unique_frac"], r["k_min"])) }



With sex alone, no one is identifiable (the smallest group holds thousands); with sex and district, still no one, because 20,000 people share 120 cells. Adding the exact birth date makes 99.4&nbsp;% of records unique &mdash; an attacker who knows a neighbour's birthday, sex and district can find their diagnosis. Generalising the date to a year leaves a residual 7.6&nbsp;% of unique records; generalising to a decade achieves k-anonymity with \( k \) well above 1 (the smallest group holds six people), at the cost of information: a model that needed age to the year has lost it. That trade-off between utility and privacy is unavoidable and is the engineer's to make explicitly, with the data owner. Note too that k-anonymity protects identity, not attributes: if every record in a group has the same diagnosis, knowing someone is in the group reveals the diagnosis (the problem addressed by the stronger notions of <em>l-diversity</em> and <em>differential privacy</em>, which are beyond this course).

<div class="box key"><h4>Exercise 3</h4><p>Reduce the population to <code>N = 2000</code> &mdash; a realistic single-hospital study &mdash; and repeat both chunks. Which generalisation is now needed to make every record share its quasi-identifiers with at least five others? Small datasets are more re-identifiable, not less.</p></div>

### 7.3 Dataset shift and shortcut learning

A classifier learns whatever regularity in the training sample predicts the label, whether or not that regularity has anything to do with the disease. When a regularity is an accident of how the data were collected, the classifier has learned a <strong>shortcut</strong>, and it fails as soon as the data are collected differently.<sup class="c"><a href="#r25">[25]</a></sup> The documented case is instructive: a network trained to detect pneumonia on chest radiographs from several hospitals learned to recognise <em>which hospital</em> an image came from &mdash; from the position of metal tokens and the image style &mdash; and used the hospital's pneumonia prevalence as a proxy, achieving high internal accuracy and poor external accuracy.<sup class="c"><a href="#r26">[26]</a></sup> The simulation below reproduces the mechanism with the breast-cancer data: a "scanner identifier" is added that, in the training hospital, happens to correlate with the label because suspicious cases were routed to one machine &mdash; a plausible workflow artefact.



In [ ]:
# Shortcut learning: a spurious feature that predicts the label in training but not at deployment.
set.seed(0)
tri <- sample(nrow(bc), round(0.7 * nrow(bc)))
Xtr <- X[tri, ]; Xte <- X[-tri, ]; ytr <- y[tri]; yte <- y[-tri]
mal_tr <- ytr == "malignant"; mal_te <- yte == "malignant"
# Training hospital: 97% of malignant cases on scanner 1, 97% of benign on scanner 0.
scanner_tr   <- ifelse(runif(length(ytr)) < 0.97, as.integer(mal_tr), as.integer(!mal_tr))
scanner_same <- ifelse(runif(length(yte)) < 0.97, as.integer(mal_te), as.integer(!mal_te))  # same hospital
scanner_diff <- sample(0:1, length(yte), replace = TRUE)                                    # new hospital
ctl <- rpart.control(maxdepth = 3, cp = 0, xval = 0)
tree_shortcut <- rpart(lab ~ ., data.frame(Xtr, scanner = scanner_tr, lab = ytr), method = "class", control = ctl)
tree_honest   <- rpart(lab ~ ., data.frame(Xtr, lab = ytr), method = "class", control = ctl)
cat("Root split of the model with the scanner feature:", as.character(tree_shortcut$frame$var[1]), "\n\n")
acc <- function(m, d, truth) mean(predict(m, d, type = "class") == truth)
data.frame(
  `training accuracy`            = c(acc(tree_shortcut, data.frame(Xtr, scanner = scanner_tr), ytr), acc(tree_honest, Xtr, ytr)),
  `new cases, same hospital`     = c(acc(tree_shortcut, data.frame(Xte, scanner = scanner_same), yte), acc(tree_honest, Xte, yte)),
  `new cases, different hospital`= c(acc(tree_shortcut, data.frame(Xte, scanner = scanner_diff), yte), acc(tree_honest, Xte, yte)),
  row.names = c("model WITH scanner feature", "model WITHOUT scanner feature"), check.names = FALSE) |> round(3)



The model given the scanner identifier put it at the root of its tree: it is the single most predictive feature in the training hospital. Its accuracy is excellent on training data and on new cases from the same hospital &mdash; the shortcut still works there &mdash; and collapses at the new hospital (from about 0.98 to 0.88), where the scanner carries no information. The model without the spurious feature is essentially unchanged across hospitals. Nothing in the training-hospital numbers warned of the failure; only testing on data from the deployment setting revealed it. This is the general lesson of external validation, and the reason the sepsis model of Section 6 failed.<sup class="c"><a href="#r20">[20]</a></sup>

### 7.4 Label bias: the algorithm learns the label, not the intention

In 2019 Obermeyer and colleagues analysed a commercial algorithm used by health systems covering some 200 million people in the United States to identify patients with complex needs for extra care.<sup class="c"><a href="#r27">[27]</a></sup> The algorithm predicted future <em>healthcare cost</em> as a proxy for <em>health need</em>. Because less money is spent on Black patients than on White patients with the same level of illness &mdash; for reasons of access, trust and discrimination &mdash; the algorithm ranked Black patients as lower-risk than equally sick White patients: at a given risk score, Black patients had on average 26.3&nbsp;% more chronic conditions. No racial variable was used; the bias entered through the choice of label. Reformulating the label as a measure of illness rather than cost reduced the disparity by 84&nbsp;%. The simulation below reproduces the mechanism in its simplest form.



In [ ]:
# Label bias: two groups with identical illness, unequal spending; a model trained on cost ranks them differently.
set.seed(0); n_per <- 5000
illness <- rgamma(2 * n_per, shape = 2, scale = 1)          # true health need, same distribution in both groups
group   <- rep(c("A","B"), each = n_per)
spend   <- ifelse(group == "A", 1.0, 0.7)                   # group B receives 30% less care for the same illness
cost    <- illness * spend * rlnorm(2 * n_per, 0, 0.3)      # observed cost (the label the algorithm is trained on)
flagged <- cost >= quantile(cost, 0.8)                      # flag the top 20% by predicted cost
tab <- data.frame(
  mean_illness            = tapply(illness, group, mean),
  mean_cost               = tapply(cost, group, mean),
  share_flagged           = tapply(flagged, group, mean),
  mean_illness_of_flagged = tapply(illness[flagged], group[flagged], mean))
round(tab, 3)



The two groups are, by construction, equally ill (mean health need about 2.0 in both). Because one receives less care, its recorded cost is lower, and an algorithm trained to predict cost &mdash; perfectly, in this simulation &mdash; flags 27&nbsp;% of group A but only 13&nbsp;% of group B, and the group-B members it does flag are <em>sicker</em> than those flagged in group A (illness 4.4 against 3.7). The algorithm is accurate at its task; the task was the wrong one. Choosing the label is the most consequential modelling decision and is rarely treated as such.

### 7.5 The regulatory and ethical framework

Three bodies of rules bear on an engineer who builds learning systems for health, and the course expects you to know what each covers.

**Data protection** (Section 7.1): Chile's Laws 19.628 and 20.584, the new Law 21.719,<sup class="c"><a href="#r22">[22]</a></sup> the GDPR and HIPAA govern the <em>collection and processing</em> of personal data &mdash; lawful basis, consent, purpose limitation, minimisation, the rights of the data subject, and the obligation to notify breaches.

**Medical-device regulation** governs the <em>system</em>. Software that informs or makes a clinical decision is a medical device (software as a medical device, SaMD). In the United States the FDA regulates it, and has published an action plan and good-machine-learning-practice principles for AI-based devices;<sup class="c"><a href="#r18">[18]</a></sup> in the European Union the Medical Device Regulation (2017/745) applies and, since 2024, the <strong>Artificial Intelligence Act</strong> classifies AI used in medical devices as <em>high-risk</em>, with obligations of risk management, data governance, documentation, human oversight, accuracy and robustness.<sup class="c"><a href="#r28">[28]</a></sup> In Chile the Instituto de Salud Pública (ISP) registers medical devices, and a national AI policy and bill have been in development since 2021.

**Ethical guidance** governs the <em>practice</em>. The World Health Organization's 2021 guidance on the ethics and governance of AI for health sets out six principles &mdash; protect autonomy; promote human well-being, safety and the public interest; ensure transparency, explainability and intelligibility; foster responsibility and accountability; ensure inclusiveness and equity; promote AI that is responsive and sustainable.<sup class="c"><a href="#r29">[29]</a></sup> These are not slogans; each maps to an engineering requirement demonstrated in this section: equity to the label-bias analysis of 7.4, robustness to the external validation of 7.3, transparency to the readable rules of Section 3.5, and autonomy and privacy to the de-identification of 7.2.

### 7.6 Generated content, misinformation and the public debate on risk

The lecture closes with an exercise: pairs of faces, some photographed and some generated, and the question "which is authentic?". Most people perform at or near chance, and the models have improved since the slides were made. Synthetic media that depict real people saying or doing things they did not &mdash; <strong>deepfakes</strong> &mdash; were identified as a threat to privacy, democratic deliberation and security as soon as the generative methods became practical,<sup class="c"><a href="#r37">[37]</a></sup> and the relevant point for a health engineer is that the same techniques generate synthetic medical images, synthetic clinical notes and synthetic voices, each of which can be used for research (augmenting a training set, protecting privacy) or for fraud (fabricating evidence of disease for insurance, impersonating a clinician). Detection is an arms race that the generators are winning; provenance &mdash; cryptographic signing of genuine content at the point of capture &mdash; is the direction regulation is taking.

The public debate about AI risk spans a wide range, from the concrete harms documented in Sections 7.2&ndash;7.5 to statements by laboratory leaders, including the chief executive of OpenAI quoted in the lecture, that advanced AI could pose risks on the scale of pandemics or nuclear war. This course does not adjudicate that debate. It asks you to notice that the concrete harms are the ones you will be in a position to cause or prevent in your professional life, that they are well understood, and that each has an engineering countermeasure you have now seen in code.

<div class="box warn"><h4>The professional's obligation</h4><p>A system that is 95&nbsp;% accurate on its development data and is deployed without external validation, on data obtained without a lawful basis, with a label that encodes an inequity, satisfies every technical criterion of this unit and violates every ethical one. The methods of this course tell you how to build such systems; this section tells you that building them well includes refusing to deploy them before the questions of Sections 7.1&ndash;7.4 have been answered. Workshop 1 of the course asks you to find and present a case in which AI endangers some facet of human life; the mechanisms of this section &mdash; re-identification, shortcut learning, label bias &mdash; are where to look.</p></div>

## 8. Summary

You should now be able to explain, without notes:

- Why "artificial intelligence" has no single definition, and what the rational-agent definition offers an engineer that the Turing test does not.
- The sequence McCulloch&ndash;Pitts &rarr; perceptron &rarr; <em>Perceptrons</em> &rarr; backpropagation &rarr; AlexNet, what each contributed, and why the two winters happened.
- The difference between knowledge that is programmed (expert systems such as MYCIN) and knowledge that is learned, and why the latter made data the central concern of the field.
- The four approaches &mdash; symbolic, connectionist, statistical, evolutionary &mdash; with one medical example each; the difference between an algorithm and a model; the ANI/AGI/ASI tiers and the reactive / limited-memory / theory-of-mind / self-aware taxonomy, and where every system in this course sits (limited memory).
- What generative AI is (text, image, video), why GPT is generative and BERT is not, and what new failure modes it brings (fluent falsehoods; synthetic media).
- The five stages of the pattern-recognition pipeline and which two this course concentrates on.
- The data matrix \( \mathbf{X} \) (\( n \times p \)) and label vector \( \mathbf{y} \); the five data types and why nominal variables must be one-hot encoded.
- How to turn an image or a signal into a feature vector, and why a pattern is a region of feature space.
- The nearest-centroid classifier: how it is trained, how it decides, what its decision boundary looks like, and how its accuracy depends on the size of the training sample.
- The three properties of the training sample (sample &ne; population; labels define the task; size bounds learning) and the failure mode associated with each (dataset shift and shortcuts; label bias; overfitting).
- Re-identification by quasi-identifiers, k-anonymity, and the Chilean and international rules that govern health data and AI medical devices.

## 9. Self-assessment

<details><summary>1. Russell and Norvig's four definitions of AI differ along two axes. Name them, and say in which quadrant a retinal-screening system belongs and why.</summary>Human versus rational, and thought versus action. A screening system is a rational agent (acting rationally): it perceives an image and acts &mdash; outputs a referral decision &mdash; so as to maximise a measurable objective; it makes no claim to think like, or appear as, a human.</details>

<details><summary>2. What did Minsky and Papert prove in 1969, and why did it not prevent today's neural networks?</summary>That a single-layer perceptron cannot represent functions such as XOR that are not linearly separable. Multi-layer networks can represent them, and backpropagation (1986) made multi-layer networks trainable; what was missing in 1969 was a training algorithm and the compute to run it, not a theoretical possibility.</details>

<details><summary>3. MYCIN performed at specialist level and was never deployed. Give two reasons, and say which one still applies to AI systems today.</summary>Legal liability for machine recommendations, integration into clinical workflow (interactive sessions took longer than a consult), the computing infrastructure of the 1970s, and the brittleness and maintenance cost of hand-written rules. Liability and workflow integration remain the main barriers today; brittleness has been transformed rather than solved (learned systems are brittle under dataset shift).</details>

<details><summary>4. In the chunks of Section 3.5, the learned tree (95.5%) agrees with the biopsy more often than the hand-written rule (89.0%). Give two reasons why that comparison is unfair to the hand-written rule.</summary>The tree's agreement was measured on the same cases it was fitted to, so it is optimistically biased; and the tree searched all nine features and all thresholds while the rule used two features with round-number thresholds. A fair comparison would evaluate both on held-out cases (Supplement S1).</details>

<details><summary>5. A dataset codes tumour stage as I = 1, II = 2, III = 3, IV = 4 and blood group as O = 1, A = 2, B = 3, AB = 4. Which coding is defensible and which is not, and what should be done instead?</summary>Stage is ordinal, so integer codes preserve the order and are defensible if one accepts equal spacing. Blood group is nominal; the integer coding invents an order and distances. It should be one-hot encoded into four binary columns.</details>

<details><summary>6. In Figure 5 the 1,000-sample signals were reduced to two features. Name one piece of information lost in that reduction and describe how you would find out whether the loss matters.</summary>The waveform shape of each beat (for example, QRS width or T-wave morphology) is lost; only timing is kept. Whether it matters is decided empirically: train classifiers on both representations and compare their performance on held-out data for the task at hand.</details>

<details><summary>7. Compute by hand: nearest-centroid classifier with centroids \( \mu_B = (12, 2) \) and \( \mu_M = (19, 9) \); classify \( \mathbf{x} = (16, 6) \).</summary>\( d_B = \sqrt{16 + 16} = \sqrt{32} \approx 5.66 \); \( d_M = \sqrt{9 + 9} = \sqrt{18} \approx 4.24 \). The case is classified malignant. (It is on the malignant side of the bisector, which passes through the midpoint (15.5, 5.5).)</details>

<details><summary>8. In Figure 8 the accuracy stops improving after a few dozen training cases. Is this evidence that more data would be useless for this problem?</summary>No. It is evidence that more data would be useless for <em>this classifier</em>, whose straight-line boundary cannot represent the overlap between classes. A more flexible classifier would continue to benefit from more data (and would need it to avoid overfitting).</details>

<details><summary>9. A hospital releases a dataset with name and identity number removed, keeping date of birth, sex and comuna. Is it anonymised? What is the minimal change that would make it 5-anonymous, and what does that cost?</summary>No; those three quasi-identifiers make most records unique (Section 7.2). Generalising date of birth to year or decade, and possibly comuna to region, until every combination appears at least five times would achieve 5-anonymity, at the cost of age resolution and geographical detail for any downstream model.</details>

<details><summary>10. In the shortcut-learning simulation, which single number would have revealed the problem before deployment, and why do the training-hospital numbers fail to reveal it?</summary>The accuracy on data from the deployment hospital (external validation). In the training hospital the scanner identifier genuinely predicts the label, so every internal measurement &mdash; training accuracy, held-out accuracy from the same hospital &mdash; remains high.</details>

<details><summary>11. The algorithm in Obermeyer et al. used no racial variable. How did racial bias enter it, and what was the remedy?</summary>Through the label: healthcare cost was used as a proxy for health need, and cost is systematically lower for Black patients at the same level of illness. Replacing the cost label by a measure of illness (number of active chronic conditions) reduced the disparity by 84&nbsp;%.</details>

<details><summary>12. "The algorithm decided to deny the claim." Rewrite the sentence using the vocabulary of Section 1.2 and explain what changed.</summary>"A model, produced by a learning algorithm from historical claims data, predicted a high probability of fraud and the system acted on the prediction." The rewrite makes explicit that the behaviour was learned from data (so the training sample and its label can be examined), that a threshold and an action policy were chosen by people, and that the deployed object is a model, not a procedure.</details>

<details><summary>13. Deep Blue is classified as "reactive" and a trained spam filter as "limited memory", yet Deep Blue searched millions of positions and the spam filter only counts words. Why is the filter placed in the higher tier?</summary>The taxonomy sorts by use of the past, not by computational effort. Deep Blue's evaluation was fixed at design time and it carried nothing from one game to the next; the filter's parameters were learned from stored past examples and change if it is retrained. Learning from stored experience is what defines the limited-memory tier.</details>

<details><summary>14. Which of the WHO's six principles does external validation serve, and which does k-anonymity serve?</summary>External validation serves safety, robustness (well-being and public interest) and, when stratified by subgroup, inclusiveness and equity. k-anonymity serves the protection of autonomy and privacy.</details>

## Glossary

<dl class="gloss">
<dt>Artificial intelligence (AI)</dt><dd>The study and construction of systems that perceive their environment and act so as to achieve their goals; historically, the attempt to make machines perform tasks that require intelligence when performed by people.</dd>
<dt>Anonymisation / pseudonymisation</dt><dd>Irreversible removal of the link between a record and a person / replacement of identifiers by a code whose key is kept separately (reversible; still personal data under the GDPR).</dd>
<dt>Classifier</dt><dd>A function that maps a feature vector to a class label or to a probability distribution over classes; its parameters are learned from a training sample.</dd>
<dt>Connectionism</dt><dd>The approach to AI in which knowledge is stored implicitly in the weights of networks of simple units and acquired by training.</dd>
<dt>Dataset shift</dt><dd>A difference between the distribution of the training data and that of the data encountered in deployment; the main cause of degraded performance of deployed models.</dd>
<dt>Deepfake</dt><dd>Synthetic image, audio or video that depicts a real person saying or doing something they did not; produced by generative models.</dd>
<dt>ELIZA effect</dt><dd>The tendency of users to attribute understanding or intention to a system that merely manipulates symbols; named after Weizenbaum's 1966 chatbot.</dd>
<dt>Expert system</dt><dd>A symbolic AI program that encodes the knowledge of a human expert as rules and reasons with them; MYCIN is the medical prototype.</dd>
<dt>Generative AI</dt><dd>Models trained on large collections of content that produce new content of the same kind (text, images, video, molecules); large language models and diffusion models are the main families.</dd>
<dt>Feature / feature vector</dt><dd>A measured or computed property of an object / the fixed-length vector of all such properties that represents the object to the classifier.</dd>
<dt>k-anonymity</dt><dd>The property of a dataset in which every combination of quasi-identifier values is shared by at least k records.</dd>
<dt>Label (target, response)</dt><dd>The known class or value associated with each training object; what the classifier learns to predict.</dd>
<dt>Machine learning</dt><dd>The sub-field of AI in which a system's behaviour is learned from data rather than programmed.</dd>
<dt>Model vs algorithm</dt><dd>An algorithm is a fixed procedure; a model is an object with parameters learned from data by a learning algorithm. The model is what is deployed.</dd>
<dt>Nearest-centroid classifier</dt><dd>The classifier that assigns an object to the class whose mean feature vector is closest; its decision boundaries are hyperplanes.</dd>
<dt>Nominal / ordinal / continuous</dt><dd>Data types: unordered categories / ordered categories without fixed spacing / real-valued measurements. The type determines the admissible operations.</dd>
<dt>One-hot encoding</dt><dd>Representation of a nominal variable with k levels as k binary indicator columns, so that no order or distance between levels is implied.</dd>
<dt>Pattern recognition</dt><dd>The discipline concerned with the automatic classification of objects (images, signals, measurements) into categories.</dd>
<dt>Perceptron</dt><dd>Rosenblatt's 1958 learning machine: a threshold unit whose weights are adjusted from labelled examples; the ancestor of neural networks.</dd>
<dt>Quasi-identifier</dt><dd>An attribute that is not an identifier by itself but, combined with others, can single out a person (date of birth, sex, postal code).</dd>
<dt>Rational agent</dt><dd>A system that perceives through sensors and acts through actuators so as to maximise the expected achievement of its objective; the working definition of AI in modern textbooks.</dd>
<dt>Shortcut learning</dt><dd>A model's reliance on a spurious regularity of the training data (such as a hospital marker) that predicts the label there but not elsewhere.</dd>
<dt>Training sample (training set)</dt><dd>The labelled data from which a classifier's parameters are learned; a sample from, not the whole of, the population the classifier will be applied to.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">McCarthy J, Minsky ML, Rochester N, Shannon CE. A proposal for the Dartmouth Summer Research Project on Artificial Intelligence, August 31, 1955. <em>AI Magazine</em>. 2006;27(4):12&ndash;14. <a href="https://doi.org/10.1609/aimag.v27i4.1904">doi:10.1609/aimag.v27i4.1904</a></li>
<li id="r2">Turing AM. Computing machinery and intelligence. <em>Mind</em>. 1950;LIX(236):433&ndash;460. <a href="https://doi.org/10.1093/mind/LIX.236.433">doi:10.1093/mind/LIX.236.433</a></li>
<li id="r3">Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. Chapter 1. <a href="https://doi.org/10.1109/MIS.2022.3186559">doi:10.1109/MIS.2022.3186559</a> (review of the 4th edition in <em>IEEE Intelligent Systems</em>; the book itself has no DOI)</li>
<li id="r4">Singhal K, Azizi S, Tu T, et al. Large language models encode clinical knowledge. <em>Nature</em>. 2023;620:172&ndash;180. <a href="https://doi.org/10.1038/s41586-023-06291-2">doi:10.1038/s41586-023-06291-2</a></li>
<li id="r5">McCulloch WS, Pitts W. A logical calculus of the ideas immanent in nervous activity. <em>Bull Math Biophys</em>. 1943;5:115&ndash;133. <a href="https://doi.org/10.1007/BF02478259">doi:10.1007/BF02478259</a></li>
<li id="r6">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386&ndash;408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r7">Minsky M, Papert S. <em>Perceptrons: An Introduction to Computational Geometry</em>. MIT Press; 1969. Reissued 2017. <a href="https://doi.org/10.7551/mitpress/11301.001.0001">doi:10.7551/mitpress/11301.001.0001</a></li>
<li id="r8">Shortliffe EH, Buchanan BG. A model of inexact reasoning in medicine. <em>Math Biosci</em>. 1975;23(3&ndash;4):351&ndash;379. <a href="https://doi.org/10.1016/0025-5564(75)90047-4">doi:10.1016/0025-5564(75)90047-4</a></li>
<li id="r9">Rumelhart DE, Hinton GE, Williams RJ. Learning representations by back-propagating errors. <em>Nature</em>. 1986;323:533&ndash;536. <a href="https://doi.org/10.1038/323533a0">doi:10.1038/323533a0</a></li>
<li id="r10">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84&ndash;90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r11">Silver D, Huang A, Maddison CJ, et al. Mastering the game of Go with deep neural networks and tree search. <em>Nature</em>. 2016;529:484&ndash;489. <a href="https://doi.org/10.1038/nature16961">doi:10.1038/nature16961</a></li>
<li id="r12">Jumper J, Evans R, Pritzel A, et al. Highly accurate protein structure prediction with AlphaFold. <em>Nature</em>. 2021;596:583&ndash;589. <a href="https://doi.org/10.1038/s41586-021-03819-2">doi:10.1038/s41586-021-03819-2</a></li>
<li id="r13">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861&ndash;870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r14">Jain AK, Duin RPW, Mao J. Statistical pattern recognition: a review. <em>IEEE Trans Pattern Anal Mach Intell</em>. 2000;22(1):4&ndash;37. <a href="https://doi.org/10.1109/34.824819">doi:10.1109/34.824819</a></li>
<li id="r15">Alpaydin E, Kaynak C. Cascading classifiers. <em>Kybernetika</em>. 1998;34(4):369&ndash;374. UCI Optical Recognition of Handwritten Digits dataset. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r16">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402&ndash;2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r17">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115&ndash;118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r18">Benjamens S, Dhunnoo P, Meskó B. The state of artificial intelligence-based FDA-approved medical devices and algorithms: an online database. <em>npj Digit Med</em>. 2020;3:118. <a href="https://doi.org/10.1038/s41746-020-00324-0">doi:10.1038/s41746-020-00324-0</a></li>
<li id="r19">Hannun AY, Rajpurkar P, Haghpanahi M, et al. Cardiologist-level arrhythmia detection and classification in ambulatory electrocardiograms using a deep neural network. <em>Nat Med</em>. 2019;25:65&ndash;69. <a href="https://doi.org/10.1038/s41591-018-0268-3">doi:10.1038/s41591-018-0268-3</a></li>
<li id="r20">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065&ndash;1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r21">Stokes JM, Yang K, Swanson K, et al. A deep learning approach to antibiotic discovery. <em>Cell</em>. 2020;180(4):688&ndash;702. <a href="https://doi.org/10.1016/j.cell.2020.01.021">doi:10.1016/j.cell.2020.01.021</a></li>
<li id="r22">Ley N° 21.719, que regula la protección y el tratamiento de los datos personales y crea la Agencia de Protección de Datos Personales. Diario Oficial de la República de Chile, 13 December 2024; entry into force 1 December 2026. Biblioteca del Congreso Nacional. <a href="https://www.bcn.cl/leychile">https://www.bcn.cl/leychile</a> (search "Ley 21719"; consolidated text of Law 19.628 also available there)</li>
<li id="r23">Sweeney L. k-anonymity: a model for protecting privacy. <em>Int J Uncertain Fuzziness Knowl-Based Syst</em>. 2002;10(5):557&ndash;570. <a href="https://doi.org/10.1142/S0218488502001648">doi:10.1142/S0218488502001648</a></li>
<li id="r24">Rocher L, Hendrickx JM, de Montjoye Y-A. Estimating the success of re-identifications in incomplete datasets using generative models. <em>Nat Commun</em>. 2019;10:3069. <a href="https://doi.org/10.1038/s41467-019-10933-3">doi:10.1038/s41467-019-10933-3</a></li>
<li id="r25">Geirhos R, Jacobsen J-H, Michaelis C, et al. Shortcut learning in deep neural networks. <em>Nat Mach Intell</em>. 2020;2:665&ndash;673. <a href="https://doi.org/10.1038/s42256-020-00257-z">doi:10.1038/s42256-020-00257-z</a></li>
<li id="r26">Zech JR, Badgeley MA, Liu M, Costa AB, Titano JJ, Oermann EK. Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. <em>PLoS Med</em>. 2018;15(11):e1002683. <a href="https://doi.org/10.1371/journal.pmed.1002683">doi:10.1371/journal.pmed.1002683</a></li>
<li id="r27">Obermeyer Z, Powers B, Vogeli C, Mullainathan S. Dissecting racial bias in an algorithm used to manage the health of populations. <em>Science</em>. 2019;366(6464):447&ndash;453. <a href="https://doi.org/10.1126/science.aax2342">doi:10.1126/science.aax2342</a></li>
<li id="r28">Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence (Artificial Intelligence Act). <em>Official Journal of the European Union</em>. 2024;L 2024/1689. <a href="https://eur-lex.europa.eu/eli/reg/2024/1689/oj">https://eur-lex.europa.eu/eli/reg/2024/1689/oj</a></li>
<li id="r29">World Health Organization. <em>Ethics and governance of artificial intelligence for health: WHO guidance</em>. Geneva: WHO; 2021. ISBN 978-92-4-002920-0. <a href="https://www.who.int/publications/i/item/9789240029200">https://www.who.int/publications/i/item/9789240029200</a></li>
<li id="r30">Hintze A. Understanding the four types of AI, from reactive robots to self-aware beings. <em>The Conversation</em>. 14 November 2016. <a href="https://theconversation.com/understanding-the-four-types-of-ai-from-reactive-robots-to-self-aware-beings-67616">theconversation.com/&hellip;-67616</a> (popular-science article; the taxonomy has no primary research source)</li>
<li id="r31">Hebb DO. <em>The Organization of Behavior: A Neuropsychological Theory</em>. New York: Wiley; 1949. For the modern formulation of the rule see Brown TH, Kairiss EW, Keenan CL. Hebbian synapses: biophysical mechanisms and algorithms. <em>Annu Rev Neurosci</em>. 1990;13:475&ndash;511. <a href="https://doi.org/10.1146/annurev.ne.13.030190.002355">doi:10.1146/annurev.ne.13.030190.002355</a></li>
<li id="r32">Newell A, Simon H. The logic theory machine &mdash; a complex information processing system. <em>IRE Trans Inf Theory</em>. 1956;2(3):61&ndash;79. <a href="https://doi.org/10.1109/TIT.1956.1056797">doi:10.1109/TIT.1956.1056797</a></li>
<li id="r33">Weizenbaum J. ELIZA &mdash; a computer program for the study of natural language communication between man and machine. <em>Commun ACM</em>. 1966;9(1):36&ndash;45. <a href="https://doi.org/10.1145/365153.365168">doi:10.1145/365153.365168</a></li>
<li id="r34">Ferrucci D, Brown E, Chu-Carroll J, et al. Building Watson: an overview of the DeepQA project. <em>AI Magazine</em>. 2010;31(3):59&ndash;79. <a href="https://doi.org/10.1609/aimag.v31i3.2303">doi:10.1609/aimag.v31i3.2303</a></li>
<li id="r35">Vaswani A, Shazeer N, Parmar N, et al. Attention is all you need. <em>Advances in Neural Information Processing Systems 30 (NeurIPS 2017)</em>. <a href="https://doi.org/10.48550/arXiv.1706.03762">doi:10.48550/arXiv.1706.03762</a></li>
<li id="r36">Devlin J, Chang M-W, Lee K, Toutanova K. BERT: pre-training of deep bidirectional transformers for language understanding. <em>Proc NAACL-HLT 2019</em>. 2019:4171&ndash;4186. <a href="https://doi.org/10.18653/v1/N19-1423">doi:10.18653/v1/N19-1423</a></li>
<li id="r37">Chesney R, Citron D. Deep fakes: a looming challenge for privacy, democracy, and national security. <em>Calif Law Rev</em>. 2019;107(6):1753&ndash;1820. <a href="https://doi.org/10.15779/Z38RV0D15J">doi:10.15779/Z38RV0D15J</a></li>
<li id="r38">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package (ships the original Wisconsin breast-cancer data used here). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r39">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r40">Therneau TM, Atkinson EJ. An Introduction to Recursive Partitioning Using the RPART Routines. Mayo Foundation technical report; R package <code>rpart</code>. <a href="https://doi.org/10.32614/CRAN.package.rpart">doi:10.32614/CRAN.package.rpart</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. &mdash; Chapters 1&ndash;2 for the definitions and history of this unit; the standard reference of the field.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. &mdash; Chapter 1 for the statistical view of pattern recognition. Free PDF from the author's site.</li>
<li>Duda RO, Hart PE, Stork DG. <em>Pattern Classification</em>. 2nd ed. Wiley; 2001. &mdash; The classic pattern-recognition text; Chapter 1 defines the pipeline of Section 4.2.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. &mdash; The R companion to the whole course; free PDF at statlearning.com.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. &mdash; The reference for the R modelling functions used throughout this edition.</li>
<li>Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25:44&ndash;56. &mdash; A readable survey of medical applications and their evidence base.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>mlbench package documentation &mdash; <a href="https://cran.r-project.org/package=mlbench">cran.r-project.org/package=mlbench</a>; the breast-cancer and other benchmark datasets used in this course.</li>
<li>UCI Machine Learning Repository &mdash; <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; original source of the breast-cancer and digits data.</li>
<li>Biblioteca del Congreso Nacional de Chile, Ley Chile &mdash; <a href="https://www.bcn.cl/leychile">bcn.cl/leychile</a>; consolidated texts of Laws 19.628, 20.584 and 21.719.</li>
<li>FDA, Artificial Intelligence and Machine Learning in Software as a Medical Device &mdash; <a href="https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-and-machine-learning-software-medical-device">fda.gov</a>.</li>
<li>WHO guidance on ethics and governance of AI for health &mdash; <a href="https://www.who.int/publications/i/item/9789240029200">who.int</a>.</li>
</ul>
